# 22. Парная single-phase абляция

Этот ноутбук сохраняет проверенную реализацию исходного эксперимента и имена его
артефактов. Запускайте **Restart Kernel and Run All Cells** после выполнения всех
предыдущих пронумерованных ноутбуков.

Все входы, кроме исходных raw-данных из `config/raw_sources.json`, создаются внутри
этого проекта. Результаты записываются в `outputs/`, а модели — в `checkpoints/`.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


# Single-phase FT ablation после обогащения RRUFF SG

Честный парный эксперимент для проверки влияния многофазных real-спектров.

Обе модели стартуют из одного `pretrain_v2_full_best.pt`, используют одинаковые source-wise
outer folds, одну и ту же однофазную validation-часть и одинаковый максимальный бюджет
optimizer steps. Отличается только train:

- `all_real_control`: все 3475 real-строк;
- `single_phase_only`: только строки `phase_count == 1`.

Подбор режима, числа шагов и elements-threshold выполняется только на inner split. Outer
validation не участвует в выборе. Основной результат — paired difference на одинаковых
однофазных OOF-строках. Многофазный outer holdout считается отдельно как диагностический,
но не участвует в подборе.

Synthetic replay во время FT отсутствует. Текущий synthetic pretrain уже полностью
однофазный, поэтому повторять pretrain для этой абляции не требуется.

In [2]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset

SEED = 42

def reset_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

reset_seeds()
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
device_name = torch.cuda.get_device_name(0) if AMP else 'CPU'
print('device:', DEVICE, '|', device_name)

device: cuda | NVIDIA GeForce RTX 3060 Ti


In [3]:
# ---------- настройки эксперимента ----------
MODE = 'cv'  # 'cv' — полный nested 5-fold; 'quick' — один fold и короткий подбор

BASE = PROJECT_ROOT
DATA = BASE / 'data' / 'preprocessed'
CKPT_PRE = BASE / 'checkpoints' / 'pretrain_v2_full_best.pt'
OUT = BASE / 'outputs'
CKPT_DIR = BASE / 'checkpoints'

RUN_SUFFIX = '_quick' if MODE == 'quick' else ''
OUTPUT_PREFIX = f'ft_single_phase_ablation_with_rruff_sg{RUN_SUFFIX}'
FINAL_MODEL_PREFIX = f'ft_single_phase_only_with_rruff_sg{RUN_SUFFIX}'

GRID_N = 4096
W = 3
BATCH = 64
OUTER_FOLDS = 5
INNER_VAL_FRAC = 0.20
MAX_REFERENCE_EPOCHS = 30 if MODE == 'cv' else 6
PATIENCE_CHECKS = 7 if MODE == 'cv' else 2
MIN_CHECKS = 5 if MODE == 'cv' else 2
WARMUP_FRAC = 0.10
CLIP = 1.0
EL_THRESHOLDS = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

SYSTEMS = ['triclinic', 'monoclinic', 'orthorhombic', 'tetragonal',
           'trigonal', 'hexagonal', 'cubic']
IMPUTE_LAMBDA = 1.5406
W_LAT, W_VOL, W_SG, W_SYS, W_EL, W_ANG = 1.0, 0.5, 1.0, 1.0, 1.0, 2.0

CANDIDATES = [
    {'name': 'head_only', 'lr_backbone': 0.0, 'lr_head': 8e-5, 'wd': 1e-4},
    {'name': 'conservative', 'lr_backbone': 3e-6, 'lr_head': 3e-5, 'wd': 1e-4},
    {'name': 'balanced', 'lr_backbone': 8e-6, 'lr_head': 5e-5, 'wd': 1e-4},
    {'name': 'stronger', 'lr_backbone': 1.5e-5, 'lr_head': 1e-4, 'wd': 3e-4},
]

ARMS = ['all_real_control', 'single_phase_only']

print('mode:', MODE)
print('candidates:', [c['name'] for c in CANDIDATES])
print('arms:', ARMS)
print('synthetic replay: OFF')

mode: cv
candidates: ['head_only', 'conservative', 'balanced', 'stronger']
arms: ['all_real_control', 'single_phase_only']
synthetic replay: OFF


In [4]:
# ---------- данные и аудит однофазного поднабора ----------
stats = json.loads((OUT / 'pretrain_stats.json').read_text(encoding='utf-8'))
VOCAB = stats['vocab']
EL_IDX = {e: i for i, e in enumerate(VOCAB)}
LAT_MEAN = np.asarray(stats['lat_mean'], dtype=np.float64)
LAT_STD = np.asarray(stats['lat_std'], dtype=np.float64)
VOL_MEAN, VOL_STD = stats['vol_mean'], stats['vol_std']

index = pd.read_parquet(DATA / 'index_preprocessed.parquet')
N_TOTAL = len(index)
X_MM = np.memmap(DATA / 'X_intensity.f16', dtype=np.float16, mode='r',
                 shape=(N_TOTAL, GRID_N))
M_MM = np.memmap(DATA / 'M_mask.u8', dtype=np.uint8, mode='r',
                 shape=(N_TOTAL, GRID_N))
row_of = dict(zip(index['sample_id'], index['row_idx']))

def to_list(v):
    if isinstance(v, str):
        try:
            parsed = json.loads(v)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    if isinstance(v, (list, tuple, np.ndarray)):
        return [e for e in v if isinstance(e, str)]
    return []

def prepare_pool(filename, source_key):
    frame = pd.read_parquet(BASE / 'data' / 'clean' / filename).copy()
    frame['dataset_source'] = source_key
    frame['row_idx'] = frame['sample_id'].map(row_of)
    assert frame['row_idx'].notna().all(), f'{source_key}: отсутствует row_idx'

    frame['phase_count'] = pd.to_numeric(frame['phase_count'], errors='coerce').fillna(0).astype(int)
    assert frame['phase_count'].ge(1).all(), (
        f'{source_key}: FT-пул содержит phase_count=0/NaN; это unknown, а не отдельный класс')
    if 'is_single_phase' in frame:
        expected = frame['phase_count'].eq(1)
        observed = frame['is_single_phase'].fillna(False).astype(bool)
        assert expected.equals(observed), f'{source_key}: phase_count и is_single_phase расходятся'

    frame['primary_wavelength'] = frame['primary_wavelength'].fillna(IMPUTE_LAMBDA)
    if 'secondary_wavelength' not in frame:
        frame['secondary_wavelength'] = np.nan
    frame['crystal_system'] = frame['crystal_system'].replace({'rhombohedral': 'trigonal'})

    abc = frame[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
    ang = frame[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
    ca, cb, cg = (np.cos(np.radians(ang[:, i])) for i in range(3))
    volume_term = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
    frame['V'] = abc.prod(1) * np.sqrt(np.clip(volume_term, 1e-12, None))
    frame['lat6'] = list(np.hstack([np.log(abc), ang]))
    frame['elements'] = frame['elements_list'].apply(to_list)

    if source_key == 'rruff':
        assert 'rruff_mineral_name' in frame.columns
        assert frame['rruff_mineral_name'].notna().all()
        frame['conn_key'] = ('rruff_mineral|' +
                             frame['rruff_mineral_name'].astype(str).str.casefold())
    else:
        composition = frame['phase_compositions'].fillna('').astype(str)
        frame['conn_key'] = composition + '|' + frame['lattice_a'].round(2).astype(str)
    return frame.reset_index(drop=True)

ft_rruff = prepare_pool('ft_pool_rruff_with_rruff_sg.parquet', 'rruff')
ft_opxrd = prepare_pool('ft_pool_opxrd.parquet', 'opxrd')
ft_all = pd.concat([ft_rruff, ft_opxrd], ignore_index=True)
ft_single = ft_all[ft_all['phase_count'].eq(1)].reset_index(drop=True)

def coverage(frame):
    return {
        'rows': len(frame),
        'groups': frame['conn_key'].nunique(),
        'lattice': int(frame['lattice_a'].notna().sum()),
        'system': int(frame['crystal_system'].isin(SYSTEMS).sum()),
        'space_group': int(frame['spacegroup_number'].notna().sum()),
        'elements': int(frame['elements'].map(len).gt(0).sum()),
    }

audit_rows = []
for subset_name, subset in [('all_real', ft_all), ('single_phase', ft_single)]:
    for source_name in ['combined', 'rruff', 'opxrd']:
        part = subset if source_name == 'combined' else subset[subset['dataset_source'].eq(source_name)]
        audit_rows.append({'subset': subset_name, 'source': source_name, **coverage(part)})
audit_df = pd.DataFrame(audit_rows)
print(audit_df.to_string(index=False))

assert len(ft_rruff) == 1359 and ft_rruff['phase_count'].eq(1).all()
assert len(ft_opxrd) == 2116
assert len(ft_all) == 3475 and len(ft_single) == 2406
assert ft_single.query("dataset_source == 'opxrd'")['spacegroup_number'].notna().sum() == 0
assert ft_single.query("dataset_source == 'opxrd'")['elements'].map(len).gt(0).sum() == 0
print('Аудит пройден: single-phase opXRD участвует только в lattice/system heads.')

      subset   source  rows  groups  lattice  system  space_group  elements
    all_real combined  3475    1584     2653    2684         1675      2253
    all_real    rruff  1359     806     1298    1329         1105      1357
    all_real    opxrd  2116     778     1355    1355          570       896
single_phase combined  2406    1469     2345    2376         1105      1357
single_phase    rruff  1359     806     1298    1329         1105      1357
single_phase    opxrd  1047     663     1047    1047            0         0
Аудит пройден: single-phase opXRD участвует только в lattice/system heads.


In [5]:
# ---------- Dataset и маски частичных меток ----------
def build_labels(frame):
    lam1 = frame['primary_wavelength'].to_numpy(np.float32)
    lam2 = frame['secondary_wavelength'].to_numpy(np.float32)
    lam = np.stack([lam1 / 1.54, np.nan_to_num(lam2) / 1.54,
                    np.isfinite(lam2).astype(np.float32)], axis=1)

    lat6 = np.stack(frame['lat6'].to_numpy())
    has_lat = ~np.isnan(lat6).any(axis=1)
    latm = has_lat.astype(np.float32)
    lat6 = (lat6 - LAT_MEAN) / LAT_STD

    vol_raw = frame['V'].to_numpy(np.float64)
    vol = (np.log(vol_raw) - VOL_MEAN) / VOL_STD

    sg_raw = frame['spacegroup_number'].to_numpy(float)
    sgm = np.isfinite(sg_raw).astype(np.float32)
    sg = np.nan_to_num(sg_raw, nan=1.0).astype(np.int64) - 1

    sysmap = {s: i for i, s in enumerate(SYSTEMS)}
    sys_raw = frame['crystal_system'].map(sysmap)
    sysm = sys_raw.notna().to_numpy(np.float32)
    sys_ = sys_raw.fillna(0).to_numpy(np.int64)

    n = len(frame)
    el = np.zeros((n, len(VOCAB)), np.float32)
    elm = np.zeros(n, np.float32)
    for i, els in enumerate(frame['elements']):
        if els:
            elm[i] = 1.0
            for e in els:
                j = EL_IDX.get(e)
                if j is not None:
                    el[i, j] = 1.0

    return dict(lam=lam, lat6=lat6.astype(np.float32), latm=latm,
                vol=vol.astype(np.float32), volm=latm.copy(),
                sg=sg, sgm=sgm, sys_=sys_, sysm=sysm, el=el, elm=elm)

class SpecDS(Dataset):
    def __init__(self, frame):
        self.row = frame['row_idx'].to_numpy(np.int64)
        self.L = build_labels(frame)

    def __len__(self):
        return len(self.row)

    def __getitem__(self, i):
        x = np.empty((2, GRID_N), np.float32)
        x[0] = X_MM[self.row[i]]
        x[1] = M_MM[self.row[i]]
        L = self.L
        return (torch.from_numpy(x), torch.from_numpy(L['lam'][i]),
                torch.from_numpy(L['lat6'][i]), torch.tensor(L['latm'][i]),
                torch.tensor(L['sg'][i]), torch.tensor(L['sgm'][i]),
                torch.tensor(L['sys_'][i]), torch.tensor(L['sysm'][i]),
                torch.from_numpy(L['el'][i]), torch.tensor(L['elm'][i]),
                torch.tensor(L['vol'][i]), torch.tensor(L['volm'][i]),
                torch.tensor(i, dtype=torch.int64))

def make_loader(frame, shuffle, seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        SpecDS(frame.reset_index(drop=True)), batch_size=BATCH, shuffle=shuffle,
        pin_memory=AMP, num_workers=0, drop_last=False, generator=generator,
    )

def n_batches(frame):
    return max(1, math.ceil(len(frame) / BATCH))

In [6]:
# ---------- XRDNetV2: архитектура идентична последнему pretrain/FT ----------
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.n1 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, padding=1, bias=False)
        self.n2 = nn.GroupNorm(8, cout)
        if cin == cout and stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(
                nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                nn.GroupNorm(8, cout),
            )

    def forward(self, x):
        h = F.gelu(self.n1(self.conv1(x)))
        h = self.n2(self.conv2(h))
        return F.gelu(h + self.skip(x))

class XRDNetV2(nn.Module):
    def __init__(self, n_el, w=3):
        super().__init__()
        c = [32 * w, 48 * w, 64 * w, 96 * w, 128 * w, 192 * w, 256 * w]
        self.stem = nn.Sequential(
            nn.Conv1d(2, c[0], 15, padding=7, bias=False),
            nn.GroupNorm(8, c[0]), nn.GELU(),
        )
        self.blocks = nn.Sequential(*[
            ResBlock(c[i], c[i + 1], stride=2) for i in range(6)
        ])
        self.lam_mlp = nn.Sequential(
            nn.Linear(3, 16 * w), nn.GELU(), nn.Linear(16 * w, 16 * w),
        )
        self.trunk = nn.Sequential(
            nn.Linear(c[-1] + 16 * w, 512 * w), nn.GELU(),
            nn.Linear(512 * w, 512 * w), nn.GELU(),
        )
        self.head_lat = nn.Linear(512 * w, 6)
        self.head_vol = nn.Linear(512 * w, 1)
        self.head_sg = nn.Linear(512 * w, 230)
        self.head_sys = nn.Linear(512 * w, 7)
        self.head_el = nn.Linear(512 * w, n_el)

    def forward(self, x, lam):
        f = self.blocks(self.stem(x))
        w = F.adaptive_avg_pool1d(x[:, 1:2], f.shape[-1]).clamp_min(1e-3)
        pooled = (f * w).sum(-1) / w.sum(-1)
        z = self.trunk(torch.cat([pooled, self.lam_mlp(lam)], dim=1))
        return dict(
            lat=self.head_lat(z), vol=self.head_vol(z).squeeze(-1),
            sg=self.head_sg(z), sys=self.head_sys(z), el=self.head_el(z),
        )

def load_pretrained():
    model = XRDNetV2(len(VOCAB), w=W).to(DEVICE)
    state = torch.load(CKPT_PRE, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    return model

probe = load_pretrained()
print('parameters: %.2f M' % (sum(p.numel() for p in probe.parameters()) / 1e6))
del probe
if AMP:
    torch.cuda.empty_cache()

parameters: 11.27 M


### Данные и промежуточные вычисления

In [7]:
# ---------- loss, построчные OOF-предсказания и метрики ----------

### Функция `masked_l1`

In [8]:
def masked_l1(pred, target, mask):
    m = mask > 0
    return (F.smooth_l1_loss(pred[m], target[m])
            if m.any() else pred.new_zeros(()))

### Функция `masked_ce`

In [9]:
def masked_ce(logits, target, mask):
    m = mask > 0
    return (F.cross_entropy(logits[m], target[m].long())
            if m.any() else logits.new_zeros(()))

### Функция `compute_losses`

In [10]:
def compute_losses(out, batch12):
    (_, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm) = batch12
    m = latm > 0
    if m.any():
        loss_len = F.smooth_l1_loss(out['lat'][m][:, :3], lat[m][:, :3])
        loss_ang = F.smooth_l1_loss(out['lat'][m][:, 3:], lat[m][:, 3:])
    else:
        loss_len = loss_ang = out['lat'].new_zeros(())

    me = elm > 0
    loss_el = (F.binary_cross_entropy_with_logits(out['el'][me], el[me])
               if me.any() else out['el'].new_zeros(()))
    return (
        W_LAT * loss_len + W_ANG * loss_ang + W_VOL * masked_l1(out['vol'], vol, volm)
        + W_SG * masked_ce(out['sg'], sg, sgm)
        + W_SYS * masked_ce(out['sys'], sys_, sysm)
        + W_EL * loss_el
    )

### Функция `predict_rows`

In [11]:
@torch.no_grad()
def predict_rows(model, frame, arm, fold, eval_set, el_threshold=0.5):
    frame = frame.reset_index(drop=True)
    loader = make_loader(frame, shuffle=False, seed=SEED + 1000 + fold)
    model.eval()
    rows = []
    for batch in loader:
        local_idx = batch[-1].numpy()
        data = [b.to(DEVICE, non_blocking=AMP) for b in batch[:-1]]
        x, lam, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm = data
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            out = model(x, lam)

        lat_pred = out['lat'].float().cpu().numpy() * LAT_STD + LAT_MEAN
        lat_true = lat.float().cpu().numpy() * LAT_STD + LAT_MEAN
        lat_pred[:, :3] = np.exp(lat_pred[:, :3])
        lat_true[:, :3] = np.exp(lat_true[:, :3])
        vol_pred = np.exp(out['vol'].float().cpu().numpy() * VOL_STD + VOL_MEAN)
        vol_true = np.exp(vol.float().cpu().numpy() * VOL_STD + VOL_MEAN)
        sg_logits = out['sg'].float().cpu()
        sys_logits = out['sys'].float().cpu()
        el_prob = torch.sigmoid(out['el']).float().cpu().numpy()
        el_true = el.float().cpu().numpy()

        for j, source_idx in enumerate(local_idx):
            meta = frame.iloc[int(source_idx)]
            has_lat = float(latm[j]) > 0
            has_sg = float(sgm[j]) > 0
            has_sys = float(sysm[j]) > 0
            has_el = float(elm[j]) > 0

            pred_set = frozenset(np.where(el_prob[j] > el_threshold)[0])
            true_set = frozenset(np.where(el_true[j] > 0.5)[0]) if has_el else frozenset()
            err_len = (np.abs(lat_pred[j, :3] - lat_true[j, :3])
                       if has_lat else np.full(3, np.nan))
            ape_len = (100 * err_len / np.maximum(lat_true[j, :3], 1e-6)
                       if has_lat else np.full(3, np.nan))
            pred_sg = int(sg_logits[j].argmax().item())
            true_sg = int(sg[j].item())
            pred_sys = int(sys_logits[j].argmax().item())
            true_sys = int(sys_[j].item())

            rows.append({
                'fold': fold, 'arm': arm, 'eval_set': eval_set,
                'sample_id': meta['sample_id'],
                'dataset_source': meta['dataset_source'],
                'phase_count': int(meta['phase_count']),
                'conn_key': meta['conn_key'],
                'latm': float(has_lat), 'sgm': float(has_sg),
                'sysm': float(has_sys), 'elm': float(has_el),
                'pred_sg': pred_sg + 1 if has_sg else np.nan,
                'true_sg': true_sg + 1 if has_sg else np.nan,
                'sg_ok': float(has_sg and pred_sg == true_sg),
                'sg_top5': float(has_sg and true_sg in sg_logits[j].topk(5).indices.tolist()),
                'pred_system': SYSTEMS[pred_sys] if has_sys else None,
                'true_system': SYSTEMS[true_sys] if has_sys else None,
                'sys_ok': float(has_sys and pred_sys == true_sys),
                'mae_a': err_len[0], 'mae_b': err_len[1], 'mae_c': err_len[2],
                'mape_a': ape_len[0], 'mape_b': ape_len[1], 'mape_c': ape_len[2],
                'mae_ang': (float(np.abs(lat_pred[j, 3:] - lat_true[j, 3:]).mean())
                            if has_lat else np.nan),
                'mae_vol': (abs(vol_pred[j] - vol_true[j]) if has_lat else np.nan),
                'mape_vol': (100 * abs(vol_pred[j] - vol_true[j]) / max(vol_true[j], 1e-6)
                             if has_lat else np.nan),
                'el_tp': len(pred_set & true_set) if has_el else np.nan,
                'el_fp': len(pred_set - true_set) if has_el else np.nan,
                'el_fn': len(true_set - pred_set) if has_el else np.nan,
                'el_exact': float(pred_set == true_set) if has_el else np.nan,
                'pred_elements': json.dumps([VOCAB[k] for k in sorted(pred_set)]),
                'true_elements': json.dumps([VOCAB[k] for k in sorted(true_set)]),
                'elements_threshold': el_threshold,
            })
    model.train()
    return pd.DataFrame(rows)

### Функция `aggregate_rows`

In [12]:
def aggregate_rows(rows):
    result = {'n': len(rows)}
    valid = rows[rows['sgm'] > 0]
    result.update(
        sg_n=len(valid),
        sg_acc=valid['sg_ok'].mean() if len(valid) else np.nan,
        sg_top5=valid['sg_top5'].mean() if len(valid) else np.nan,
    )
    valid = rows[rows['sysm'] > 0]
    result.update(
        sys_n=len(valid),
        sys_acc=valid['sys_ok'].mean() if len(valid) else np.nan,
    )
    valid = rows[rows['latm'] > 0]
    result['lat_n'] = len(valid)
    if len(valid):
        for key in ['mae_a', 'mae_b', 'mae_c', 'mae_ang', 'mae_vol']:
            result[key] = valid[key].mean()
        result['mae_a_med'] = valid['mae_a'].median()
        for key in ['mape_a', 'mape_b', 'mape_c', 'mape_vol']:
            result[key + '_med'] = valid[key].median()
    valid = rows[rows['elm'] > 0]
    result['el_n'] = len(valid)
    if len(valid):
        tp, fp, fn = valid[['el_tp', 'el_fp', 'el_fn']].sum()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        result['el_f1_micro'] = 2 * precision * recall / max(precision + recall, 1e-9)
        result['el_exact'] = valid['el_exact'].mean()
    return result

### Функция `metrics_by_scope`

In [13]:
def metrics_by_scope(rows):
    output = []
    for scope in ['combined', 'rruff', 'opxrd']:
        part = rows if scope == 'combined' else rows[rows['dataset_source'].eq(scope)]
        output.append({'scope': scope, **aggregate_rows(part)})
    return output

### Функция `safe_metric`

In [14]:
def safe_metric(metrics, key, default=0.0):
    value = metrics.get(key, np.nan)
    return default if value is None or not np.isfinite(value) else float(value)

### Функция `selection_score`

In [15]:
def selection_score(metrics):
    # Подбор выполняется только на single-phase inner validation.
    return (
        safe_metric(metrics, 'sys_acc') + safe_metric(metrics, 'el_f1_micro')
        + 0.50 * safe_metric(metrics, 'sg_acc')
        + 0.25 * safe_metric(metrics, 'sg_top5')
        - 0.05 * safe_metric(metrics, 'mae_a')
        - 0.02 * safe_metric(metrics, 'mae_ang')
    )

In [16]:
# ---------- обучение с бюджетом в optimizer steps ----------
HEAD_PREFIXES = ('head_', 'trunk', 'lam_mlp')

def make_optimizer(model, cfg):
    backbone = [p for n, p in model.named_parameters() if not n.startswith(HEAD_PREFIXES)]
    heads = [p for n, p in model.named_parameters() if n.startswith(HEAD_PREFIXES)]
    if cfg['lr_backbone'] == 0:
        for p in backbone:
            p.requires_grad_(False)
        groups = [{'params': heads, 'lr': cfg['lr_head']}]
    else:
        groups = [
            {'params': backbone, 'lr': cfg['lr_backbone']},
            {'params': heads, 'lr': cfg['lr_head']},
        ]
    return torch.optim.AdamW(groups, weight_decay=cfg['wd'])

def make_scheduler(optimizer, total_steps):
    warmup = max(1, int(total_steps * WARMUP_FRAC))
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: (step / warmup if step < warmup else
                      0.5 * (1 + math.cos(math.pi * min(
                          (step - warmup) / max(total_steps - warmup, 1), 1))))
    )

def train_updates(model, frame, cfg, n_steps, seed):
    assert len(frame) > 0 and n_steps > 0
    reset_seeds(seed)
    loader = make_loader(frame, shuffle=True, seed=seed)
    iterator = iter(loader)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, n_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    model.train()
    successful_steps = 0
    while successful_steps < n_steps:
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)
        data = [b.to(DEVICE, non_blocking=AMP) for b in batch[:-1]]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            loss = compute_losses(model(data[0], data[1]), data)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        # При AMP overflow optimizer.step пропускается. Такой проход не
        # считается optimizer step и не должен сдвигать scheduler.
        if scaler.get_scale() >= scale_before:
            scheduler.step()
            successful_steps += 1
    return model

def tune_candidate(train_frame, common_val_single, cfg, reference_batches, seed):
    reset_seeds(seed)
    model = load_pretrained()
    loader = make_loader(train_frame, shuffle=True, seed=seed)
    iterator = iter(loader)
    max_steps = MAX_REFERENCE_EPOCHS * reference_batches
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, max_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)

    best_score = -np.inf
    best_reference_epoch = 1
    best_state = None
    stale = 0
    steps_done = 0

    for reference_epoch in range(1, MAX_REFERENCE_EPOCHS + 1):
        model.train()
        successful_in_interval = 0
        while successful_in_interval < reference_batches:
            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                batch = next(iterator)
            data = [b.to(DEVICE, non_blocking=AMP) for b in batch[:-1]]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                loss = compute_losses(model(data[0], data[1]), data)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= scale_before:
                scheduler.step()
                steps_done += 1
                successful_in_interval += 1

        pred = predict_rows(
            model, common_val_single, arm='inner', fold=0,
            eval_set='single_phase', el_threshold=0.5,
        )
        metrics = aggregate_rows(pred)
        score = selection_score(metrics)
        if score > best_score + 1e-4:
            best_score = score
            best_reference_epoch = reference_epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if reference_epoch >= MIN_CHECKS and stale >= PATIENCE_CHECKS:
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    best_threshold = 0.5
    best_threshold_score = -np.inf
    best_metrics = None
    for threshold in EL_THRESHOLDS:
        pred = predict_rows(
            model, common_val_single, arm='inner', fold=0,
            eval_set='single_phase', el_threshold=threshold,
        )
        metrics = aggregate_rows(pred)
        score = selection_score(metrics)
        if score > best_threshold_score:
            best_threshold_score = score
            best_threshold = threshold
            best_metrics = metrics

    return {
        'model': model,
        'best_reference_epoch': best_reference_epoch,
        'best_steps': best_reference_epoch * reference_batches,
        'el_threshold': best_threshold,
        'score': best_threshold_score,
        'metrics': best_metrics,
        'steps_examined': steps_done,
    }

In [17]:
# ---------- paired nested GroupKFold ----------
def source_outer_splits(frame):
    splitter = GroupKFold(n_splits=OUTER_FOLDS)
    return list(splitter.split(frame, groups=frame['conn_key']))

def source_inner_split(frame, seed):
    # Split строится на полном outer-train источника, но качество варианта
    # оценивается по покрытию single-phase validation, общей для двух arms.
    best = None
    best_coverage = -1
    for attempt in range(30):
        splitter = GroupShuffleSplit(
            n_splits=1, test_size=INNER_VAL_FRAC, random_state=seed + attempt,
        )
        tr_idx, va_idx = next(splitter.split(frame, groups=frame['conn_key']))
        va = frame.iloc[va_idx]
        va = va[va['phase_count'].eq(1)]
        current = (
            va['lattice_a'].notna().sum()
            + va['crystal_system'].isin(SYSTEMS).sum()
            + va['elements'].map(len).gt(0).sum()
            + va['spacegroup_number'].notna().sum()
        )
        if current > best_coverage:
            best = (tr_idx, va_idx)
            best_coverage = current
    assert best is not None
    return best

rruff_outer = source_outer_splits(ft_rruff)
opxrd_outer = source_outer_splits(ft_opxrd)
fold_ids = range(OUTER_FOLDS) if MODE == 'cv' else range(1)

tuning_rows = []
fold_rows = []
oof_frames = []
t0 = time.time()

for fold0 in fold_ids:
    fold = fold0 + 1
    rr_tr_idx, rr_va_idx = rruff_outer[fold0]
    op_tr_idx, op_va_idx = opxrd_outer[fold0]
    rr_train = ft_rruff.iloc[rr_tr_idx].reset_index(drop=True)
    rr_val = ft_rruff.iloc[rr_va_idx].reset_index(drop=True)
    op_train = ft_opxrd.iloc[op_tr_idx].reset_index(drop=True)
    op_val = ft_opxrd.iloc[op_va_idx].reset_index(drop=True)

    outer_train_all = pd.concat([rr_train, op_train], ignore_index=True)
    outer_train_single = outer_train_all[outer_train_all['phase_count'].eq(1)].reset_index(drop=True)
    outer_val_all = pd.concat([rr_val, op_val], ignore_index=True)
    outer_val_single = outer_val_all[outer_val_all['phase_count'].eq(1)].reset_index(drop=True)
    outer_val_multi = outer_val_all[outer_val_all['phase_count'].gt(1)].reset_index(drop=True)

    rr_in_tr, rr_in_va = source_inner_split(rr_train, SEED + 100 * fold)
    op_in_tr, op_in_va = source_inner_split(op_train, SEED + 200 * fold)
    inner_train_all = pd.concat(
        [rr_train.iloc[rr_in_tr], op_train.iloc[op_in_tr]], ignore_index=True,
    )
    inner_val_all = pd.concat(
        [rr_train.iloc[rr_in_va], op_train.iloc[op_in_va]], ignore_index=True,
    )
    inner_train_single = inner_train_all[inner_train_all['phase_count'].eq(1)].reset_index(drop=True)
    inner_val_single = inner_val_all[inner_val_all['phase_count'].eq(1)].reset_index(drop=True)

    assert set(inner_train_all['conn_key']).isdisjoint(set(inner_val_all['conn_key']))
    assert set(outer_train_all['conn_key']).isdisjoint(set(outer_val_all['conn_key']))
    assert len(inner_val_single) > 0 and len(outer_val_single) > 0

    inner_reference_batches = n_batches(inner_train_all)
    outer_reference_batches = n_batches(outer_train_all)

    print(f'\n===== OUTER FOLD {fold}/{len(list(fold_ids))} =====')
    print('outer train all/single:', len(outer_train_all), len(outer_train_single))
    print('outer val single/multi:', len(outer_val_single), len(outer_val_multi))
    print('inner train all/single:', len(inner_train_all), len(inner_train_single))
    print('reference batches:', inner_reference_batches)

    zero = load_pretrained()
    for eval_set, eval_frame in [('single_phase', outer_val_single),
                                 ('multiphase_diagnostic', outer_val_multi)]:
        if len(eval_frame) == 0:
            continue
        pred = predict_rows(zero, eval_frame, 'zero_shot', fold, eval_set, 0.5)
        oof_frames.append(pred)
        for item in metrics_by_scope(pred):
            fold_rows.append({
                'fold': fold, 'arm': 'zero_shot', 'eval_set': eval_set,
                'candidate': 'pretrain', 'reference_epochs': 0,
                'optimizer_steps': 0, 'el_threshold': 0.5, **item,
            })
    del zero
    if AMP:
        torch.cuda.empty_cache()

    train_frames = {
        'all_real_control': inner_train_all,
        'single_phase_only': inner_train_single,
    }
    outer_train_frames = {
        'all_real_control': outer_train_all,
        'single_phase_only': outer_train_single,
    }

    for arm_index, arm in enumerate(ARMS):
        print('\n  ARM:', arm, '| inner rows:', len(train_frames[arm]))
        candidate_results = []
        for candidate_index, cfg in enumerate(CANDIDATES):
            run_seed = SEED + 10_000 * fold + 100 * arm_index + candidate_index
            print('    tuning', cfg['name'])
            tuned = tune_candidate(
                train_frames[arm], inner_val_single, cfg,
                inner_reference_batches, run_seed,
            )
            tuning_rows.append({
                'fold': fold, 'arm': arm, 'candidate': cfg['name'],
                'best_reference_epoch': tuned['best_reference_epoch'],
                'best_steps': tuned['best_steps'],
                'el_threshold': tuned['el_threshold'],
                'inner_score': tuned['score'],
                'train_rows': len(train_frames[arm]),
                'reference_train_rows': len(inner_train_all),
                **tuned['metrics'],
            })
            candidate_results.append((tuned['score'], cfg, tuned))
            print('      score=%.4f ref_epoch=%d steps=%d threshold=%.2f' % (
                tuned['score'], tuned['best_reference_epoch'], tuned['best_steps'],
                tuned['el_threshold']))
            del tuned['model']
            if AMP:
                torch.cuda.empty_cache()

        _, chosen_cfg, chosen = max(candidate_results, key=lambda x: x[0])
        chosen_ref_epoch = chosen['best_reference_epoch']
        chosen_threshold = chosen['el_threshold']
        chosen_steps = chosen_ref_epoch * outer_reference_batches
        print('    selected:', chosen_cfg['name'], '| ref_epoch:', chosen_ref_epoch,
              '| outer steps:', chosen_steps, '| threshold:', chosen_threshold)

        reset_seeds(SEED + 50_000 * fold + arm_index)
        model = load_pretrained()
        model = train_updates(
            model, outer_train_frames[arm], chosen_cfg, chosen_steps,
            seed=SEED + 50_000 * fold + arm_index,
        )

        for eval_set, eval_frame in [('single_phase', outer_val_single),
                                     ('multiphase_diagnostic', outer_val_multi)]:
            if len(eval_frame) == 0:
                continue
            pred = predict_rows(
                model, eval_frame, arm, fold, eval_set, chosen_threshold,
            )
            oof_frames.append(pred)
            for item in metrics_by_scope(pred):
                fold_rows.append({
                    'fold': fold, 'arm': arm, 'eval_set': eval_set,
                    'candidate': chosen_cfg['name'],
                    'reference_epochs': chosen_ref_epoch,
                    'optimizer_steps': chosen_steps,
                    'el_threshold': chosen_threshold, **item,
                })
            shown = aggregate_rows(pred)
            compact = {k: round(v, 4) for k, v in shown.items()
                       if k in ['sys_acc', 'sg_acc', 'sg_top5', 'el_f1_micro',
                                'mae_a', 'mae_ang'] and np.isfinite(v)}
            print('     ', eval_set, compact)
        del model
        if AMP:
            torch.cuda.empty_cache()

print(f'\nelapsed: {(time.time() - t0) / 60:.1f} min')


===== OUTER FOLD 1/5 =====
outer train all/single: 2779 1924
outer val single/multi: 482 214
inner train all/single: 2163 1481
reference batches: 34

  ARM: all_real_control | inner rows: 2163
    tuning head_only
      score=1.2402 ref_epoch=19 steps=646 threshold=0.30
    tuning conservative
      score=1.1965 ref_epoch=26 steps=884 threshold=0.20
    tuning balanced
      score=1.2402 ref_epoch=20 steps=680 threshold=0.30
    tuning stronger
      score=1.2489 ref_epoch=9 steps=306 threshold=0.30
    selected: stronger | ref_epoch: 9 | outer steps: 396 | threshold: 0.3
      single_phase {'sg_acc': np.float64(0.4402), 'sg_top5': np.float64(0.7177), 'sys_acc': np.float64(0.5537), 'mae_a': np.float64(3.71), 'mae_ang': np.float64(5.1583), 'el_f1_micro': 0.5279}
      multiphase_diagnostic {'sg_acc': np.float64(0.3836), 'sg_top5': np.float64(0.726), 'sys_acc': np.float64(0.9811), 'mae_a': np.float64(0.0889), 'mae_ang': np.float64(0.8338), 'el_f1_micro': 0.2591}

  ARM: single_phase_onl

In [18]:
# ---------- сводка, paired comparison и сохранение OOF ----------
tuning_df = pd.DataFrame(tuning_rows)
fold_df = pd.DataFrame(fold_rows)
oof_df = pd.concat(oof_frames, ignore_index=True)

metric_keys = [
    'sys_acc', 'sg_acc', 'sg_top5', 'el_f1_micro', 'el_exact',
    'mae_a', 'mae_a_med', 'mape_a_med', 'mae_b', 'mape_b_med',
    'mae_c', 'mape_c_med', 'mae_ang', 'mae_vol', 'mape_vol_med',
]

summary_rows = []
for (arm, eval_set, scope), part in fold_df.groupby(['arm', 'eval_set', 'scope']):
    for key in metric_keys:
        values = part[key].dropna() if key in part else pd.Series(dtype=float)
        summary_rows.append({
            'arm': arm, 'eval_set': eval_set, 'scope': scope, 'metric': key,
            'mean': values.mean() if len(values) else np.nan,
            'std': values.std(ddof=0) if len(values) > 1 else np.nan,
            'folds': len(values),
        })
summary_df = pd.DataFrame(summary_rows)

control = fold_df[fold_df['arm'].eq('all_real_control')]
treatment = fold_df[fold_df['arm'].eq('single_phase_only')]
paired_base = treatment.merge(
    control, on=['fold', 'eval_set', 'scope'], suffixes=('_single', '_all'),
)
comparison_rows = []
for _, row in paired_base.iterrows():
    for key in metric_keys:
        single_value = row.get(key + '_single', np.nan)
        all_value = row.get(key + '_all', np.nan)
        if np.isfinite(single_value) and np.isfinite(all_value):
            comparison_rows.append({
                'fold': row['fold'], 'eval_set': row['eval_set'], 'scope': row['scope'],
                'metric': key, 'single_phase_ft': single_value,
                'all_real_ft': all_value,
                'difference_single_minus_all': single_value - all_value,
            })
paired_long_df = pd.DataFrame(comparison_rows)
paired_summary_df = (paired_long_df.groupby(['eval_set', 'scope', 'metric'], as_index=False)
                     .agg(single_phase_ft=('single_phase_ft', 'mean'),
                          all_real_ft=('all_real_ft', 'mean'),
                          difference_single_minus_all=('difference_single_minus_all', 'mean'),
                          std_paired_difference=('difference_single_minus_all', lambda x: x.std(ddof=0)),
                          folds=('fold', 'nunique')))

print('\n===== PRIMARY: SAME SINGLE-PHASE OUTER ROWS =====')
primary = paired_summary_df[paired_summary_df['eval_set'].eq('single_phase')]
print(primary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('\nДля accuracy/F1 положительная difference_single_minus_all лучше single-phase FT;')
print('для MAE/MAPE отрицательная разница лучше single-phase FT.')

tuning_df.to_csv(OUT / f'{OUTPUT_PREFIX}_tuning.csv', index=False)
fold_df.to_csv(OUT / f'{OUTPUT_PREFIX}_fold_metrics.csv', index=False)
oof_df.to_csv(OUT / f'{OUTPUT_PREFIX}_oof_predictions.csv', index=False)
summary_df.to_csv(OUT / f'{OUTPUT_PREFIX}_cv_summary.csv', index=False)
paired_long_df.to_csv(OUT / f'{OUTPUT_PREFIX}_paired_by_fold.csv', index=False)
paired_summary_df.to_csv(OUT / f'{OUTPUT_PREFIX}_paired_summary.csv', index=False)
print('saved outputs with prefix:', OUTPUT_PREFIX)


===== PRIMARY: SAME SINGLE-PHASE OUTER ROWS =====
    eval_set    scope       metric  single_phase_ft  all_real_ft  difference_single_minus_all  std_paired_difference  folds
single_phase combined     el_exact           0.0066       0.0081                      -0.0015                 0.0018      5
single_phase combined  el_f1_micro           0.5465       0.5242                       0.0223                 0.0068      5
single_phase combined        mae_a           3.0203       3.0487                      -0.0285                 0.0593      5
single_phase combined    mae_a_med           1.9097       1.9189                      -0.0092                 0.0768      5
single_phase combined      mae_ang           5.2318       5.2766                      -0.0448                 0.0766      5
single_phase combined        mae_b           3.0631       3.0889                      -0.0259                 0.0679      5
single_phase combined        mae_c           4.0009       4.0308                 

In [19]:
# ---------- финальный checkpoint: только single-phase real FT ----------
if MODE == 'cv':
    treatment_tuning = tuning_df[tuning_df['arm'].eq('single_phase_only')]
    candidate_scores = (treatment_tuning.groupby('candidate', as_index=False)['inner_score']
                        .mean().sort_values('inner_score', ascending=False))
    final_name = candidate_scores.iloc[0]['candidate']
    final_cfg = next(c for c in CANDIDATES if c['name'] == final_name)
    chosen = treatment_tuning[treatment_tuning['candidate'].eq(final_name)]
    final_reference_epochs = int(np.clip(
        np.median(chosen['best_reference_epoch']), MIN_CHECKS, MAX_REFERENCE_EPOCHS,
    ))
    final_threshold = float(np.median(chosen['el_threshold']))
    final_reference_batches = n_batches(ft_all)
    final_steps = final_reference_epochs * final_reference_batches

    print('final config:', final_cfg)
    print('final reference epochs:', final_reference_epochs)
    print('final optimizer steps:', final_steps)
    print('elements threshold:', final_threshold)

    reset_seeds(SEED + 999_999)
    final_model = load_pretrained()
    final_model = train_updates(
        final_model, ft_single, final_cfg, final_steps, seed=SEED + 999_999,
    )
    checkpoint_path = CKPT_DIR / f'{FINAL_MODEL_PREFIX}_final.pt'
    torch.save(final_model.state_dict(), checkpoint_path)

    metadata = {
        'source': 'RRUFF + opXRD, phase_count == 1 only',
        'pretrain_checkpoint': str(CKPT_PRE),
        'config': final_cfg,
        'reference_epochs': final_reference_epochs,
        'reference_batches_from_all_real': final_reference_batches,
        'optimizer_steps': final_steps,
        'elements_threshold': final_threshold,
        'nested_cv': True,
        'paired_outer_folds': True,
        'primary_evaluation': 'same single-phase outer rows for both arms',
        'synthetic_replay': False,
        'all_real_rows_reference': len(ft_all),
        'single_phase_train_rows': len(ft_single),
        'rruff_rows': int(ft_single['dataset_source'].eq('rruff').sum()),
        'opxrd_rows': int(ft_single['dataset_source'].eq('opxrd').sum()),
    }
    (OUT / f'{FINAL_MODEL_PREFIX}_final_config.json').write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8',
    )
    print('saved:', checkpoint_path)
else:
    print('MODE=quick: final checkpoint is not created')

final config: {'name': 'stronger', 'lr_backbone': 1.5e-05, 'lr_head': 0.0001, 'wd': 0.0003}
final reference epochs: 11
final optimizer steps: 605
elements threshold: 0.3
saved: D:\Users\user\Desktop\DS_XRD_project\checkpoints\ft_single_phase_only_with_rruff_sg_final.pt


## Как интерпретировать результат

Главный файл — `outputs/ft_single_phase_ablation_with_rruff_sg_paired_summary.csv`.

Основной срез: `eval_set == single_phase`. В нём обе модели оценены на одних и тех же
однофазных outer-fold строках. Для accuracy/F1 положительная
`difference_single_minus_all` означает преимущество single-phase FT; для MAE/MAPE —
отрицательная.

`multiphase_diagnostic` не участвует в подборе и показывает цену специализации на чистых
образцах. SG и elements для однофазного opXRD отсутствуют по данным, поэтому эти метрики
в opXRD/single-phase срезе закономерно будут NaN; по этим головам однофазный эксперимент
проверяет только RRUFF.

Если single-phase FT лучше на том же однофазном holdout, многофазные строки создавали
отрицательный перенос. Если разницы нет или контроль лучше, удаление многофазных данных
не оправдано.